In [0]:
%pip install statsmodels

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose
import itertools
import warnings

# 1. Data Loading & Formatting
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv"
df = pd.read_csv(url, parse_dates=['Month'], index_col='Month')
df.index.freq = 'MS' # Set frequency to Monthly Start

In [0]:
# 2. Exploratory Data Analysis (EDA)
# Decomposing to see Trend and Seasonality
decomposition = seasonal_decompose(df['Passengers'], model='multiplicative')
decomposition.plot()
plt.show()

In [0]:
# 3. Stationarity Check (Augmented Dickey-Fuller Test)
def check_stationarity(timeseries):
    result = adfuller(timeseries)
    print(f'ADF Statistic: {result[0]}')
    print(f'p-value: {result[1]}')
    if result[1] <= 0.05:
        print("Data is stationary")
    else:
        print("Data is non-stationary (Differencing required)")

check_stationarity(df['Passengers'])

In [0]:
# 4. SARIMAX Hyperparameter Grid Search (AIC Optimization)
p = d = q = range(0, 2)
pdq = list(itertools.product(p, d, q))
seasonal_pdq = [(x[0], x[1], x[2], 12) for x in list(itertools.product(p, d, q))]

warnings.filterwarnings("ignore")
best_aic = float("inf")
best_params = None

for param in pdq:
    for param_seasonal in seasonal_pdq:
        try:
            temp_model = sm.tsa.statespace.SARIMAX(df['Passengers'],
                                                   order=param,
                                                   seasonal_order=param_seasonal,
                                                   enforce_stationarity=False,
                                                   enforce_invertibility=False)
            results = temp_model.fit(disp=False)
            if results.aic < best_aic:
                best_aic = results.aic
                best_params = (param, param_seasonal)
        except:
            continue

print(f'Best SARIMAX Model: {best_params[0]}x{best_params[1]} - AIC: {best_aic}')

In [0]:
# 5. Final Model Fitting & Diagnostics
model = sm.tsa.statespace.SARIMAX(df['Passengers'],
                                  order=best_params[0],
                                  seasonal_order=best_params[1])
results = model.fit()
results.plot_diagnostics(figsize=(12, 8))
plt.show()

# 6. Forecasting the next 24 Months
forecast = results.get_forecast(steps=24)
forecast_df = forecast.summary_frame()

# Plotting Results
plt.figure(figsize=(10, 5))
plt.plot(df['Passengers'], label='Historical Data')
plt.plot(forecast_df['mean'], label='Forecast', color='red')
plt.fill_between(forecast_df.index, forecast_df['mean_ci_lower'], forecast_df['mean_ci_upper'], color='pink', alpha=0.3)
plt.title('Airline Passenger Forecast (SARIMAX)')
plt.legend()
plt.show()